# Initialize Frappe Environment
Set up the Frappe environment, including site initialization and common imports like `frappe` and `hmac`.

In [ ]:
import hmac
import hashlib
import json
import frappe

# Initialize site context if not in bench console
# frappe.init(site="frappe.com")
# frappe.connect()


# Define Webhook Payload and Secret
Define the raw JSON string of the payload and the manual secret `tcYXdPOd9y1J0rYg6fEmGVPV` for local testing.

In [ ]:
# Provided secret
secret = "tcYXdPOd9y1J0rYg6fEmGVPV"

# Request payload from ARAL-00012
# Note: Razorpay signs the RAW body. 
# The log contains the JSON-encoded payload. We must ensure the strings are identical.
raw_payload_str = '{"account_id": "acc_SabAvj4ik6sbRe", "contains": ["payment_link", "order", "payment"], "created_at": 1776866180, "entity": "event", "event": "payment_link.paid", "payload": {"order": {"entity": {"amount": 150000, "amount_due": 0, "amount_paid": 150000, "attempts": 1, "checkout": null, "created_at": 1776866185, "currency": "INR", "description": null, "entity": "order", "id": "order_SgYwqTzu3Iq9eX", "notes": {"reference_docname": "AAPR-00127", "reference_doctype": "Aetas Advance Payment Receipt"}, "offer_id": null, "receipt": null, "status": "paid"}}, "payment": {"entity": {"acquirer_data": {"auth_code": null}, "amount": 150000, "amount_captured": 150000, "amount_refunded": 0, "amount_transferred": 0, "authentication": {"authentication_channel": "browser", "version": ""}, "bank": null, "base_amount": 150000, "captured": true, "card": {"emi": false, "entity": "card", "id": "card_SgYxL7VqZLCMP5", "international": false, "issuer": "DCBL", "last4": "1007", "name": "", "network": "Visa", "sub_type": "consumer", "type": "debit"}, "card_id": "card_SgYxL7VqZLCMP5", "contact": "+919887787644", "created_at": 1776866213, "currency": "INR", "description": "#SgYwkMrm6xnBEP", "email": "void@razorpay.com", "entity": "payment", "error_code": null, "error_description": null, "error_reason": null, "error_source": null, "error_step": null, "fee": 3300, "fee_bearer": "platform", "id": "pay_SgYxL7VqZLCMP5", "international": false, "invoice_id": null, "method": "card", "notes": {"reference_docname": "AAPR-00127", "reference_doctype": "Aetas Advance Payment Receipt"}, "order_id": "order_SgYwqTzu3Iq9eX", "refund_status": null, "status": "captured", "tax": 0, "vpa": null, "wallet": null}}, "payment_link": {"entity": {"accept_partial": false, "amount": 150000, "amount_paid": 150000, "cancelled_at": 0, "created_at": 1776866180, "currency": "INR", "customer": {"name": "Aetas Retail Private Limited"}, "description": "Payment for Advance Receipt AAPR-00127", "expire_by": 1776902400, "expired_at": 0, "first_min_partial_amount": 0, "id": "plink_SgYwkMrm6xnBEP", "notes": {"reference_docname": "AAPR-00127", "reference_doctype": "Aetas Advance Payment Receipt"}, "notify": {"email": false, "sms": false, "whatsapp": false}, "order_id": "order_SgYwqTzu3Iq9eX", "reference_id": "", "reminder_enable": false, "reminders": {}, "short_url": "https://rzp.io/rzp/LKojeIc2", "status": "paid", "updated_at": 1776866221, "upi_link": false, "user_id": "", "whatsapp_link": false}}}}'


# Compute Expected Webhook Signature
Use `hmac` with `sha256` to generate the signature from the raw payload and the secret. Formula: $signature = hmac\_sha256(payload, secret)$.

In [ ]:
# Compute HMAC-SHA256 signature
dig = hmac.new(
    key=secret.encode("utf-8"),
    msg=raw_payload_str.encode("utf-8"),
    digestmod=hashlib.sha256
)
computed_signature = dig.hexdigest()
print(f"Computed Signature: {computed_signature}")


# Compare Signatures and Debug Validation Logic
Compare the computed signature against the one received in the request headers (X-Razorpay-Signature) to verify validity.

In [ ]:
# Since we don't have the header here, we check if our logic matches what Frappe or our webhook.py does.
# Verification Logic:
def verify_signature(body, signature, key):
    expected_signature = hmac.new(
        key.encode("utf-8"),
        body.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()
    return hmac.compare_digest(expected_signature, signature)

# If the error is 'Invalid Webhook Signature', it means `hmac.compare_digest` failed.
# Possible reasons:
# 1. Payload was modified (e.g., json.loads then json.dumps)
# 2. Secret is incorrect
# 3. Encoding issues


# Inspect Razorpay Settings and Document State
Fetch the 'Razorpay Settings' for 'Art of Time (Colaba)' using `frappe.get_doc` to ensure the stored webhook secret matches the one used for validation.

In [ ]:
# Fetch correct boutique settings
settings = frappe.get_doc("Razorpay Settings", "Art of Time (Colaba)")
print(f"Stored Secret present: {bool(settings.webhook_secret)}")
# print(f"Stored Secret: {settings.get_password('webhook_secret')}") # Security sensitive

# Check if the boutique was resolved correctly in the webhook logic
# result = _get_boutique_from_payload(json.loads(raw_payload_str))
# print(f"Resolved Boutique: {result}")
